# Lab 3 — Multi-Hop Reasoning

**Goal:** Answer questions that require 2-3 graph traversal steps — impossible with pure RAG.

**Example:**  
Q: 'What country is the headquarters of the company that employs SpaceX's president in?'  
→ Shotwell → works_at → SpaceX → headquartered_in → Hawthorne → located_in → California → part_of → USA

RAG would struggle — it finds SpaceX info but can't chain the relationships reliably.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from knowledge_graph import KnowledgeGraph

# Build a richer graph for multi-hop demos
CORPUS = """
Alice Smith is the CTO of Acme Corp. Acme Corp is headquartered in Boston.
Bob Jones is the CEO of Acme Corp. Alice Smith reports to Bob Jones.
Acme Corp acquired DataCo in 2022. DataCo is based in Austin, Texas.
Carol White is the founder of DataCo. Carol White now works at Acme Corp.
Acme Corp is a subsidiary of MegaCorp. MegaCorp is headquartered in New York.
MegaCorp is listed on the NYSE. The NYSE is located in New York.
Boston is in Massachusetts. Massachusetts is in the United States.
Austin is in Texas. Texas is in the United States. New York is in the United States.
"""

kg = KnowledgeGraph()
triples = kg.add_text(CORPUS)
print(f'Graph: {kg.stats()}')

## 1. One-hop queries

In [ ]:
# Direct fact lookup
print('Direct neighbors of Alice Smith:')
for n in kg.neighbors('Alice Smith'):
    print(f'  [{n["predicate"]}] → {n["entity"]}')

## 2. Two-hop query

In [ ]:
# Q: Where is Alice Smith's employer headquartered?
# Hop 1: Alice → works_at / is_cto_of → Acme Corp
# Hop 2: Acme Corp → headquartered_in → Boston

print('Available predicates in graph:')
preds = sorted(set(d['predicate'] for _,_,d in kg.graph.edges(data=True)))
for p in preds:
    print(f'  {p}')

In [ ]:
# Find the predicate the LLM used for Alice's employer relationship
alice_neighbors = kg.neighbors('Alice Smith', direction='out')
print('Alice Smith outgoing edges:')
for n in alice_neighbors:
    print(f'  --[{n["predicate"]}]--> {n["entity"]}')

# Use the first employer-like edge to do a 2-hop query
employer_pred = next((n['predicate'] for n in alice_neighbors 
                      if any(w in n['predicate'] for w in ['work', 'employ', 'cto', 'at'])), None)
if employer_pred:
    employer = kg.multi_hop_query('Alice Smith', [employer_pred])
    print(f'\nAlice\'s employer: {employer}')

## 3. LLM-driven multi-hop Q&A

In [ ]:
questions = [
    'Who does Alice Smith report to?',
    'What company did Acme Corp acquire?',
    'Who founded the company that Acme Corp acquired?',
    'What parent company does Alice Smith ultimately work for?',
    'What stock exchange is Alice Smith\'s ultimate employer listed on?',
]

for q in questions:
    answer = kg.answer_question(q)
    print(f'Q: {q}')
    print(f'A: {answer}')
    print()

## 4. Shortest path for relationship discovery

In [ ]:
# How is Carol White connected to New York?
path = kg.shortest_path('Carol White', 'New York')
if path:
    print('Path from Carol White to New York:')
    print(' → '.join(path))
    print(f'Hops: {len(path) - 1}')
else:
    print('No path found')

## Exercise
Build a function `explain_path(kg, source, target)` that:
1. Finds the shortest path
2. Looks up the predicate for each edge in the path
3. Returns a human-readable sentence like: 'Alice works at Acme Corp, which is a subsidiary of MegaCorp, which is headquartered in New York'

Use an LLM to convert the path triples into a natural language sentence.

In [ ]:
# Your code here
